# Vesuvius Challenge 2025 - Topology-Aware Segmentation

## Key Changes from Baseline (0.562 → Target 0.60+)

| Component | Baseline | This Solution | Why It Matters |
|-----------|----------|---------------|----------------|
| Patch Size | 96³ | **192³** | More context for sheet separation |
| Epochs | 100 | **1200** | Topology learning happens late |
| Loss | Dice + MedialSurfaceRecall | **Dice + BCE + Supervoxel Connectivity** | Direct bridge penalty |
| β (merge emphasis) | N/A | **0.7** | Penalize bridges > holes |
| Post-processing | Frangi filter | **Erosion + CC + Selective Dilation** | Actually breaks bridges |
| Attention | None | **scSE blocks** | Better boundary features |

## Quick Start

```bash
# Test run (10% data, 20 epochs) - verify code works
!python vesuvius_train.py --test_run --data_fraction 0.1

# Full training
!python vesuvius_train.py --epochs 1200
```

In [ ]:
# Install dependencies if needed
!pip install connected-components-3d tifffile -q

In [ ]:
"""
Vesuvius Challenge 2025 - Topology-Aware 3D Segmentation
=========================================================
Key improvements:
1. 192³ patch size (captures more context)
2. Supervoxel Connectivity Loss with merge penalty emphasis (β=0.7)
3. Loss scheduling (topology loss starts after convergence)
4. Bridge-breaking post-processing
5. scSE attention blocks
"""

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm
import warnings
from scipy import ndimage
from scipy.ndimage import label as scipy_label
from typing import Optional, Tuple, List, Dict
import random

# Try to use cc3d, fall back to scipy if not available
try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy (slower but works)")

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Data paths
    DATA_ROOT = Path("/kaggle/input/vesuvius-challenge-2025")
    TRAIN_IMAGES = DATA_ROOT / "train_images"
    TRAIN_LABELS = DATA_ROOT / "train_labels"
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    
    # Model - KEY: 192³ patches
    PATCH_SIZE = (192, 192, 192)
    BATCH_SIZE = 1
    NUM_WORKERS = 2
    
    # Training
    EPOCHS = 1200  # Match baseline
    LEARNING_RATE = 0.01
    WEIGHT_DECAY = 3e-5
    
    # Loss weights
    DICE_WEIGHT = 0.4
    BCE_WEIGHT = 0.3
    TOPO_WEIGHT = 0.3
    TOPO_BETA = 0.7  # KEY: Penalize bridges MORE than holes
    
    # Loss scheduling
    TOPO_START_EPOCH = 200  # Start after basic convergence
    TOPO_WARMUP_EPOCHS = 200  # Gradually increase
    
    # Post-processing
    EROSION_ITERATIONS = 2
    MIN_COMPONENT_SIZE = 100
    THRESHOLD = 0.75
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Test mode
    TEST_RUN = False
    DATA_FRACTION = 1.0

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {config.DEVICE}")
print(f"Patch size: {config.PATCH_SIZE}")
print(f"Topology β (merge emphasis): {config.TOPO_BETA}")

In [ ]:
# ============================================================================
# CONNECTED COMPONENTS (with fallback)
# ============================================================================

def connected_components_3d(volume, connectivity=26):
    """Compute 3D connected components with cc3d or scipy fallback."""
    if USE_CC3D:
        return cc3d.connected_components(volume, connectivity=connectivity)
    else:
        if connectivity == 26:
            struct = ndimage.generate_binary_structure(3, 3)
        elif connectivity == 6:
            struct = ndimage.generate_binary_structure(3, 1)
        else:
            struct = ndimage.generate_binary_structure(3, 2)
        labeled, _ = scipy_label(volume, structure=struct)
        return labeled


def count_components(labeled):
    """Count number of connected components."""
    if USE_CC3D:
        return labeled.max()
    else:
        return labeled.max()

In [ ]:
# ============================================================================
# DATA LOADING
# ============================================================================

def load_volume(path: Path) -> np.ndarray:
    """Load 3D TIF volume."""
    im = Image.open(path)
    slices = [np.array(page) for page in ImageSequence.Iterator(im)]
    return np.stack(slices, axis=0)


class VesuviusDataset(Dataset):
    """Dataset for 3D scroll segmentation."""
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 4,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        df = pd.read_csv(csv_path)
        
        # Filter to existing volumes
        valid_ids = []
        for idx in df['id'].values:
            if (images_dir / f"{idx}.tif").exists() and (labels_dir / f"{idx}.tif").exists():
                valid_ids.append(idx)
        
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.seed(42)
            valid_ids = random.sample(valid_ids, n)
        
        self.volume_ids = valid_ids
        self.cache = {}
        print(f"Loaded {len(self.volume_ids)} volumes")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def _load_volume(self, vol_id):
        if vol_id in self.cache:
            return self.cache[vol_id]
        
        image = load_volume(self.images_dir / f"{vol_id}.tif").astype(np.float32)
        label = load_volume(self.labels_dir / f"{vol_id}.tif").astype(np.int64)
        
        # Z-score normalization
        image = (image - image.mean()) / (image.std() + 1e-8)
        
        if len(self.cache) < 5:
            self.cache[vol_id] = (image, label)
        
        return image, label
    
    def _extract_patch(self, image, label):
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd:
            image = np.pad(image, ((0, pd-d), (0, 0), (0, 0)))
            label = np.pad(label, ((0, pd-d), (0, 0), (0, 0)))
            d = pd
        if h < ph:
            image = np.pad(image, ((0, 0), (0, ph-h), (0, 0)))
            label = np.pad(label, ((0, 0), (0, ph-h), (0, 0)))
            h = ph
        if w < pw:
            image = np.pad(image, ((0, 0), (0, 0), (0, pw-w)))
            label = np.pad(label, ((0, 0), (0, 0), (0, pw-w)))
            w = pw
        
        z = random.randint(0, d - pd)
        y = random.randint(0, h - ph)
        x = random.randint(0, w - pw)
        
        return image[z:z+pd, y:y+ph, x:x+pw], label[z:z+pd, y:y+ph, x:x+pw]
    
    def _augment(self, image, label):
        # Random flips
        for axis in [0, 1, 2]:
            if random.random() > 0.5:
                image = np.flip(image, axis=axis).copy()
                label = np.flip(label, axis=axis).copy()
        
        # Random 90° rotations
        k = random.randint(0, 3)
        if k > 0:
            image = np.rot90(image, k=k, axes=(1, 2)).copy()
            label = np.rot90(label, k=k, axes=(1, 2)).copy()
        
        # Intensity augmentation
        if random.random() > 0.5:
            image = image * random.uniform(0.8, 1.2)
        if random.random() > 0.5:
            image = image + random.uniform(-0.1, 0.1)
        if random.random() > 0.5:
            image = image + np.random.normal(0, 0.05, image.shape).astype(np.float32)
        
        return image, label
    
    def __getitem__(self, idx):
        vol_id = self.volume_ids[idx // self.patches_per_volume]
        image, label = self._load_volume(vol_id)
        img_patch, lbl_patch = self._extract_patch(image, label)
        
        if self.is_train:
            img_patch, lbl_patch = self._augment(img_patch, lbl_patch)
        
        # Labels: 0=bg, 1=fg, 2=unlabeled
        foreground = (lbl_patch == 1).astype(np.float32)
        ignore_mask = (lbl_patch == 2).astype(np.float32)
        
        return {
            "image": torch.from_numpy(img_patch).unsqueeze(0),
            "label": torch.from_numpy(foreground).unsqueeze(0),
            "ignore_mask": torch.from_numpy(ignore_mask).unsqueeze(0),
        }

In [ ]:
# ============================================================================
# MODEL - 3D ResEnc U-Net with scSE attention
# ============================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv3d(in_ch, out_ch, 3, padding=1)
        self.norm = nn.InstanceNorm3d(out_ch, affine=True)
        self.act = nn.LeakyReLU(inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Concurrent Spatial + Channel Squeeze-and-Excitation."""
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        
        # Channel attention
        cse = self.cse_pool(x).view(b, c)
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(cse)))).view(b, c, 1, 1, 1)
        
        # Spatial attention
        sse = torch.sigmoid(self.sse_conv(x))
        
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=[32, 64, 128, 256, 320, 320]):
        super().__init__()
        blocks = [1, 3, 4, 6, 6, 6]
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        prev_ch = in_ch
        for i, (feat, n_blocks) in enumerate(zip(features, blocks)):
            enc = nn.Sequential(
                ConvBlock(prev_ch, feat),
                *[ResBlock(feat) for _ in range(n_blocks)]
            )
            self.encoders.append(enc)
            self.attentions.append(scSEBlock(feat))
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
            prev_ch = feat
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        features_rev = features[::-1]
        for i in range(len(features) - 1):
            self.ups.append(nn.ConvTranspose3d(features_rev[i], features_rev[i+1], 2, stride=2))
            self.dec_convs.append(ConvBlock(features_rev[i+1] * 2, features_rev[i+1]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
    
    def forward(self, x):
        # Encoder
        enc_feats = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_feats.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        # Decoder
        enc_feats = enc_feats[::-1]
        x = enc_feats[0]
        
        for i, (up, conv) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_feats[i + 1]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = conv(x)
        
        return self.final(x)

In [ ]:
# ============================================================================
# SUPERVOXEL CONNECTIVITY LOSS
# ============================================================================

class SupervoxelConnectivityLoss(nn.Module):
    """
    Asymmetric topology loss that penalizes bridges MORE than holes.
    
    β > 0.5: More penalty for bridges (merge errors)
    β < 0.5: More penalty for holes (split errors)
    
    For Vesuvius, we want β=0.7 because bridges kill VOI_score.
    """
    
    def __init__(self, beta=0.7):
        super().__init__()
        self.beta = beta
    
    def _find_critical_voxels(self, pred_binary, target_binary):
        """Find thin bridges (positive critical) and holes (negative critical)."""
        # False positives = potential bridges
        fp_mask = (pred_binary == 1) & (target_binary == 0)
        # False negatives = potential holes  
        fn_mask = (pred_binary == 0) & (target_binary == 1)
        
        # Thin FP regions are likely bridges
        if fp_mask.any():
            eroded = ndimage.binary_erosion(fp_mask, iterations=1)
            pos_critical = fp_mask & ~eroded
        else:
            pos_critical = np.zeros_like(fp_mask)
        
        # Thin FN regions are likely holes
        if fn_mask.any():
            eroded = ndimage.binary_erosion(fn_mask, iterations=1)
            neg_critical = fn_mask & ~eroded
        else:
            neg_critical = np.zeros_like(fn_mask)
        
        return pos_critical, neg_critical
    
    def forward(self, pred, target, ignore_mask=None):
        device = pred.device
        batch_size = pred.shape[0]
        total_loss = torch.tensor(0.0, device=device)
        
        pred_sigmoid = torch.sigmoid(pred)
        
        for b in range(batch_size):
            pred_np = (pred_sigmoid[b, 0].detach().cpu().numpy() > 0.5).astype(np.uint8)
            target_np = target[b, 0].cpu().numpy().astype(np.uint8)
            
            if ignore_mask is not None:
                ignore_np = ignore_mask[b, 0].cpu().numpy().astype(bool)
                pred_np[ignore_np] = 0
                target_np[ignore_np] = 0
            
            pos_crit, neg_crit = self._find_critical_voxels(pred_np, target_np)
            pos_crit_t = torch.from_numpy(pos_crit).to(device).float()
            neg_crit_t = torch.from_numpy(neg_crit).to(device).float()
            
            pred_b = pred[b, 0]
            
            # Loss on bridges (want these to be 0)
            if pos_crit_t.sum() > 0:
                pos_loss = F.binary_cross_entropy_with_logits(
                    pred_b[pos_crit_t > 0],
                    torch.zeros_like(pred_b[pos_crit_t > 0])
                )
            else:
                pos_loss = torch.tensor(0.0, device=device)
            
            # Loss on holes (want these to be 1)
            if neg_crit_t.sum() > 0:
                neg_loss = F.binary_cross_entropy_with_logits(
                    pred_b[neg_crit_t > 0],
                    torch.ones_like(pred_b[neg_crit_t > 0])
                )
            else:
                neg_loss = torch.tensor(0.0, device=device)
            
            # Asymmetric weighting
            batch_loss = self.beta * pos_loss + (1 - self.beta) * neg_loss
            total_loss = total_loss + batch_loss
        
        return total_loss / batch_size


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        return 1 - (2 * intersection + self.smooth) / (union + self.smooth)


class CombinedLoss(nn.Module):
    def __init__(self, dice_w=0.4, bce_w=0.3, topo_w=0.3, topo_beta=0.7):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w = bce_w
        self.topo_w = topo_w
        self.dice_loss = DiceLoss()
        self.topo_loss = SupervoxelConnectivityLoss(beta=topo_beta)
    
    def forward(self, pred, target, ignore_mask=None, topo_scale=1.0):
        dice = self.dice_loss(pred, target, ignore_mask)
        
        if ignore_mask is not None:
            valid = 1 - ignore_mask
            bce = F.binary_cross_entropy_with_logits(
                pred * valid, target * valid, reduction='sum'
            ) / (valid.sum() + 1e-6)
        else:
            bce = F.binary_cross_entropy_with_logits(pred, target)
        
        if topo_scale > 0:
            topo = self.topo_loss(pred, target, ignore_mask)
        else:
            topo = torch.tensor(0.0, device=pred.device)
        
        total = self.dice_w * dice + self.bce_w * bce + self.topo_w * topo_scale * topo
        
        return {"total": total, "dice": dice, "bce": bce, "topo": topo}

In [ ]:
# ============================================================================
# POST-PROCESSING
# ============================================================================

def topology_preserving_postprocess(
    prediction,
    threshold=0.75,
    erosion_iterations=2,
    min_component_size=100,
):
    """
    Break bridges while preserving sheets.
    
    1. Conservative threshold
    2. Erosion breaks thin bridges
    3. Remove small fragments
    4. Selective dilation to restore volume
    """
    binary = (prediction > threshold).astype(np.uint8)
    eroded = ndimage.binary_erosion(binary, iterations=erosion_iterations)
    
    labeled = connected_components_3d(eroded.astype(np.uint8))
    
    # Get component sizes
    if USE_CC3D:
        stats = cc3d.statistics(labeled)
        sizes = stats['voxel_counts']
    else:
        n_components = labeled.max()
        sizes = ndimage.sum(np.ones_like(labeled), labeled, range(1, n_components + 1))
    
    # Keep large components
    cleaned = np.zeros_like(binary)
    for comp_id in range(1, labeled.max() + 1):
        if sizes[comp_id - 1 if not USE_CC3D else comp_id] >= min_component_size:
            cleaned[labeled == comp_id] = 1
    
    # Selective dilation
    dilated = ndimage.binary_dilation(cleaned, iterations=erosion_iterations)
    final = dilated & binary
    
    return final.astype(np.uint8)

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

def get_topo_scale(epoch, cfg):
    """Loss scheduling: gradually introduce topology loss."""
    if epoch < cfg.TOPO_START_EPOCH:
        return 0.0
    elif epoch < cfg.TOPO_START_EPOCH + cfg.TOPO_WARMUP_EPOCHS:
        return (epoch - cfg.TOPO_START_EPOCH) / cfg.TOPO_WARMUP_EPOCHS
    return 1.0


def train_one_epoch(model, loader, criterion, optimizer, device, epoch, cfg):
    model.train()
    total_loss = 0
    topo_scale = get_topo_scale(epoch, cfg)
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        ignore = batch["ignore_mask"].to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        losses = criterion(logits, labels, ignore, topo_scale=topo_scale)
        losses["total"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
        optimizer.step()
        
        total_loss += losses["total"].item()
        pbar.set_postfix(loss=f"{losses['total'].item():.4f}", topo=f"{topo_scale:.2f}")
    
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device, cfg):
    model.eval()
    total_dice = 0
    total_dice_pp = 0
    n = 0
    
    for batch in tqdm(loader, desc="Validating"):
        images = batch["image"].to(device)
        labels = batch["label"].numpy()
        ignore = batch["ignore_mask"].numpy()
        
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        for b in range(images.shape[0]):
            prob = probs[b, 0]
            lbl = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0].astype(bool)
            lbl[ign] = 0
            
            # Raw
            pred_raw = (prob > 0.5).astype(np.uint8)
            pred_raw[ign] = 0
            dice_raw = 2 * (pred_raw & lbl).sum() / (pred_raw.sum() + lbl.sum() + 1e-6)
            
            # Post-processed
            pred_pp = topology_preserving_postprocess(prob, cfg.THRESHOLD, cfg.EROSION_ITERATIONS, cfg.MIN_COMPONENT_SIZE)
            pred_pp[ign] = 0
            dice_pp = 2 * (pred_pp & lbl).sum() / (pred_pp.sum() + lbl.sum() + 1e-6)
            
            total_dice += dice_raw
            total_dice_pp += dice_pp
            n += 1
    
    return {"dice_raw": total_dice / n, "dice_postproc": total_dice_pp / n}

In [ ]:
# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================

# Test run settings
TEST_RUN = True  # Set to False for full training
DATA_FRACTION = 0.1  # Use 10% of data for testing

if TEST_RUN:
    config.DATA_FRACTION = DATA_FRACTION
    config.EPOCHS = 20
    config.TOPO_START_EPOCH = 5
    config.TOPO_WARMUP_EPOCHS = 5
    print(f"TEST RUN: {config.DATA_FRACTION*100:.0f}% data, {config.EPOCHS} epochs")

# Create datasets
train_csv = config.DATA_ROOT / "train.csv"

if train_csv.exists():
    train_dataset = VesuviusDataset(
        train_csv, config.TRAIN_IMAGES, config.TRAIN_LABELS,
        config.PATCH_SIZE, is_train=True, data_fraction=config.DATA_FRACTION
    )
    val_dataset = VesuviusDataset(
        train_csv, config.TRAIN_IMAGES, config.TRAIN_LABELS,
        config.PATCH_SIZE, is_train=False, data_fraction=config.DATA_FRACTION,
        patches_per_volume=1
    )
    
    train_loader = DataLoader(train_dataset, config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
    val_loader = DataLoader(val_dataset, config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)
else:
    print(f"Data not found at {train_csv}")
    print("Please set config.DATA_ROOT to point to your data directory")

In [ ]:
# Create model and training components
model = ResEncUNet3D().to(config.DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

criterion = CombinedLoss(
    dice_w=config.DICE_WEIGHT,
    bce_w=config.BCE_WEIGHT,
    topo_w=config.TOPO_WEIGHT,
    topo_beta=config.TOPO_BETA
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config.LEARNING_RATE,
    momentum=0.99,
    weight_decay=config.WEIGHT_DECAY,
    nesterov=True
)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda e: (1 - e / config.EPOCHS) ** 0.9
)

In [ ]:
# Training loop
best_dice = 0

for epoch in range(config.EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, config.DEVICE, epoch, config)
    scheduler.step()
    
    if (epoch + 1) % 5 == 0 or epoch == config.EPOCHS - 1:
        val_metrics = validate(model, val_loader, config.DEVICE, config)
        
        print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
        print(f"  Train loss: {train_loss:.4f}")
        print(f"  Val Dice (raw): {val_metrics['dice_raw']:.4f}")
        print(f"  Val Dice (postproc): {val_metrics['dice_postproc']:.4f}")
        
        if val_metrics['dice_postproc'] > best_dice:
            best_dice = val_metrics['dice_postproc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_dice': best_dice,
            }, config.OUTPUT_DIR / 'best_model.pth')
            print(f"  Saved best model!")

print(f"\nTraining complete! Best Dice: {best_dice:.4f}")

## Summary: Why This Should Score Higher

| Change | Impact on Metrics | Why |
|--------|-------------------|-----|
| 192³ patches | +VOI, +TopoScore | More context prevents local merge decisions |
| Topo loss β=0.7 | **+VOI strongly** | Directly penalizes bridges |
| Loss scheduling | +stability | Prevents early training collapse |
| Bridge-breaking postproc | +VOI, +TopoScore | Removes thin connections |
| scSE attention | +SurfaceDice | Better boundary features |
| 1200 epochs | +all | Topology learning happens late |

Expected improvement: **0.443 → 0.55-0.60+**